In [1]:
import geopandas as gpd
from auxiliary_functions import *
from metrics_calculation import *
from pre_processing import *

import shutil
import dask_geopandas as dgpd
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Point, box, shape
import numpy as np

In [2]:
MAIN_PATH = "s3://wri-cities-sandbox/identifyingLandSubdivisions/data"
INPUT_PATH = f'{MAIN_PATH}/input'
CITY_INFO_PATH = f'{INPUT_PATH}/city_info'
EXTENTS_PATH = f'{CITY_INFO_PATH}/extents'
BUILDINGS_PATH = f'{INPUT_PATH}/buildings'
BLOCKS_PATH = f'{INPUT_PATH}/blocks'
ROADS_PATH = f'{INPUT_PATH}/roads'
INTERSECTIONS_PATH = f'{INPUT_PATH}/intersections'
GRIDS_PATH = f'{INPUT_PATH}/city_info/grids'
SEARCH_BUFFER_PATH = f'{INPUT_PATH}/city_info/search_buffers'
OUTPUT_PATH = f'{MAIN_PATH}/output'
OUTPUT_PATH_CSV = f'{OUTPUT_PATH}/csv'
OUTPUT_PATH_RASTER = f'{OUTPUT_PATH}/raster'

In [3]:
# Check s3 connection using AWS_PROFILE=CitiesUserPermissionSet profile 
AWS_PROFILE = 'cities'
import boto3
boto3.setup_default_session(profile_name='cities')
session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client('s3')

# export CitiesUserPermissionSet profile to use in the next cells
import os
os.environ['AWS_PROFILE'] = AWS_PROFILE

s3.list_buckets()

{'ResponseMetadata': {'RequestId': '2BAFJQ8B5Y5486CB',
  'HostId': 'N3X+y/Ac70htrasNKuOFpSCvS2GpokvOUij3+Uj0nDG7NriCK5plZs34gUp96T0CT5f5Y0nYAKE=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'N3X+y/Ac70htrasNKuOFpSCvS2GpokvOUij3+Uj0nDG7NriCK5plZs34gUp96T0CT5f5Y0nYAKE=',
   'x-amz-request-id': '2BAFJQ8B5Y5486CB',
   'date': 'Wed, 04 Mar 2026 20:15:51 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'aft-sandbox-540362055257',
   'CreationDate': datetime.datetime(2022, 9, 13, 15, 12, 20, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::aft-sandbox-540362055257'},
  {'Name': 'amplify-citiesindicatorsapi-dev-10508-deployment',
   'CreationDate': datetime.datetime(2023, 8, 30, 5, 5, 13, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::amplify-citiesindicatorsapi-dev-10508-deployment'},
  {'Name': 'cities-heat',
   'CreationDate': datetime.datetime(2023, 6, 1, 13, 22, 1, tzinfo=tzutc

In [ ]:
import os, shutil
import s3fs
import geopandas as gpd
import pandas as pd
from getting_1000_cities_search_areas import *

# --- use YOUR existing reader ---
def read_validation_set_s3(bucket_prefix="wri-cities-sandbox/identifyingLandSubdivisions/data/validation/Land_Use/102_cities",
                           base="Final_102Cities"):
    fs = s3fs.S3FileSystem(anon=False)
    suffixes = [".shp", ".dbf", ".shx", ".prj", ".cpg"]
    local_dir = "/tmp/land_use"
    os.makedirs(local_dir, exist_ok=True)

    for suf in suffixes:
        s3_path = f"{bucket_prefix}/{base}{suf}"
        local_path = os.path.join(local_dir, base + suf)
        with fs.open(s3_path, "rb") as fsrc, open(local_path, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)

    return gpd.read_file(os.path.join(local_dir, base + ".shp"))

validation_set = read_validation_set_s3()

# --- same replacements i already use ---
repl = {
    "United Republic of Tanzania": "Tanzania",
    "Pointe-Noire": "Pointe Noire",
    "Côte d'Ivoire": "Cote d Ivoire",
    "CÃ´te d'Ivoire": "Cote d Ivoire",
    "Mbour": "MBour",
    "Manzin": "Manzini",
    "Burkina-Faso":"Burkina Faso",
    'Republic of the Congo': "Republic of Congo",
    "Sierra-Leone":"Sierra Leone",
    "South-Africa": "South Africa",
    "Cã´Te-D'Ivoire" : "Cote d Ivoire",
}

vc = validation_set[["City", "Country"]].replace(repl).drop_duplicates()
vc["City"] = vc["City"].str.replace(
    r'(?<=-)([a-z])', lambda m: m.group(1).upper(), regex=True
)

vc["Country"] = vc["Country"].str.replace(
    r'(?<=-)([a-z])', lambda m: m.group(1).upper(), regex=True
)


In [38]:
vc

,City,Country
0,Ouagadougou,Burkina Faso
1,Kisumu,Kenya
2,Magaria,Niger
3,Freetown,Sierra Leone
4,Lusaka,Zambia
...,...,...
1916,Banki,Nigeria
2585,Moroni,Comoros
38512,Bogota,Colombia
38758,Belo_Horizonte,Brazil


In [39]:
import importlib.util


def make_folder_token(city_raw, country_raw):
    city_token = s3_safe_token(city_raw)
    country_token = s3_safe_token(country_raw) if country_raw else ""
    return f"{city_token}__{country_token}" if country_token else city_token

only_tokens = set(make_folder_token(r.City, r.Country) for r in vc.itertuples(index=False))
only_tokens



{'Abidjan__Cote_d_Ivoire',
 'Accra__Ghana',
 'Addis_Ababa__Ethiopia',
 'Ambovombe__Madagascar',
 'Antananarivo__Madagascar',
 'Asmara__Eritrea',
 'Bamako__Mali',
 'Bangui__Central_African_Republic',
 'Banki__Cameroon',
 'Banki__Nigeria',
 'Bata__Equatorial_Guinea',
 'Belo_Horizonte__Brazil',
 'Bissau__Guinea_Bissau',
 'Bo__Sierra_Leone',
 'Boditi__Ethiopia',
 'Bogota__Colombia',
 'Bondoukou__Cote_d_Ivoire',
 'Bouake__Cote_d_Ivoire',
 'Brazzaville__Republic_of_Congo',
 'Bujumbura__Burundi',
 'Bulawayo__Zimbabwe',
 'Bunia__Republic_of_Congo',
 'Buurhakaba__Somalia',
 'Campinas__Brazil',
 'Cape_Town__South_Africa',
 'Chimoio__Mozambique',
 'Conakry__Guinea',
 'Cotonou__Benin',
 'Dakar__Senegal',
 'Dapaong__Togo',
 'Dar_es_Salaam__Tanzania',
 'Djibouti__Djibouti',
 'Edendale__South_Africa',
 'Embu__Kenya',
 'Faya_Largeau__Chad',
 'Fomboni__Comoros',
 'Francistown__Botswana',
 'Freetown__Sierra_Leone',
 'Gaborone__Botswana',
 'Garoua__Cameroon',
 'Harare__Zimbabwe',
 'Ido_ekiti__Nigeria',
 

In [ ]:
# Inject the filter set into the imported module and run
#from getting_1000_cities_search_areas import *
#main(ONLY_FOLDER_TOKENS=True)


In [40]:
for city_name in only_tokens - set('Cape_Town__South_Africa'):
    print(city_name)

Boditi__Ethiopia
Serrekunda__Gambia
Campinas__Brazil
Tahoua__Niger
Gaborone__Botswana
Ndola__Zambia
Brazzaville__Republic_of_Congo
Niamey__Niger
Lome__Togo
Kuito__Angola
Nzerekore__Guinea
Nouakchott__Mauritania
Johannesburg__South_Africa
Banki__Cameroon
Bangui__Central_African_Republic
Buurhakaba__Somalia
N_Djamena__Cameroon
Lusaka__Zambia
Bulawayo__Zimbabwe
Pointe_Noire__Republic_of_Congo
Port_Gentil__Gabon
Whydah__Benin
Sikasso__Mali
Waku_Kungo__Angola
Timbuktu__Mali
Porto_Novo__Benin
Lome__Ghana
Accra__Ghana
Kissidougou__Guinea
Massawa__Eritrea
Ouagadougou__Burkina_Faso
Bo__Sierra_Leone
Lilongwe__Malawi
Faya_Largeau__Chad
Tenkodogo__Burkina_Faso
Worcester__South_Africa
Koudougou__Burkina_Faso
Bamako__Mali
Kpalime__Togo
Kakanda__Republic_of_Congo
Dapaong__Togo
Kapiri_Mposhi__Zambia
Garoua__Cameroon
Kigali__Rwanda
Djibouti__Djibouti
MBour__Senegal
Monrovia__Liberia
Nairobi__Kenya
Cape_Town__South_Africa
Maxixe__Mozambique
Mbeya__Tanzania
Maseru__Lesotho
Minna__Nigeria
Bunia__Republic_

In [41]:
cities_iterate = [x for x in only_tokens if x != 'Lagos__Nigeria']
for city_name in cities_iterate:

    #city_name = 'Cape_Town__South_Africa'#"Medellin__Colombia"
    YOUR_NAME = "sara"
    
    print(f"Processing {city_name}...")
    
    #produce_blocks(city_name, 'sara').compute()
    building_distance_metrics(city_name, YOUR_NAME).compute()
    building_and_intersection_metrics(city_name, YOUR_NAME).compute()
    compute_m6_m7(city_name, YOUR_NAME).compute()
    metrics_roads_intersections(city_name, YOUR_NAME).compute()
    metrics_k_blocks(city_name, YOUR_NAME).compute()




Processing Gaborone__Botswana...
Processing Ndola__Zambia...
Processing Kuito__Angola...
Processing Nouakchott__Mauritania...
Processing Bangui__Central_African_Republic...
Processing Lusaka__Zambia...
Processing Bulawayo__Zimbabwe...
Processing Waku_Kungo__Angola...
Processing Accra__Ghana...
Processing Kissidougou__Guinea...
Processing Lilongwe__Malawi...
Processing Tenkodogo__Burkina_Faso...
Processing Bamako__Mali...
Processing Kpalime__Togo...
Processing Dapaong__Togo...
Processing Kapiri_Mposhi__Zambia...
Processing Djibouti__Djibouti...
Processing Monrovia__Liberia...
Processing Nairobi__Kenya...
Processing Cape_Town__South_Africa...
Processing Mbeya__Tanzania...
Processing Moundou__Chad...
Processing Kismayo__Somalia...
Processing Chimoio__Mozambique...
Processing Mogadishu__Somalia...
Processing Luanda__Angola...
Processing Asmara__Eritrea...
Processing Ido_ekiti__Nigeria...


FileNotFoundError: An error occurred while calling the read_parquet method registered to the pandas backend.
Original Message: wri-cities-sandbox/identifyingLandSubdivisions/data/input/city_info/search_buffers/Ido_ekiti__Nigeria/Ido_ekiti__Nigeria_search_buffer.geoparquet

In [ ]:
fs = s3fs.S3FileSystem(anon=False)

import uuid

def safe_merge_on_index(left, right, right_prefix=None):
    """
    Merge right into left on index without suffix collisions.
    - Drops geometry from right
    - If right_prefix is provided, prefixes ALL right columns before merge
    - Drops any overlaps to avoid MergeError from duplicate suffixes
    """
    if right is None:
        return left

    right2 = right.drop(columns="geometry", errors="ignore")

    if right_prefix:
        right2 = right2.rename(columns={c: f"{right_prefix}{c}" for c in right2.columns})

    overlap = [c for c in right2.columns if c in left.columns]
    if overlap:
        right2 = right2.drop(columns=overlap, errors="ignore")

    return left.merge(right2, left_index=True, right_index=True, how="left")


def write_parquet_to_s3_atomic(gdf, out_s3_uri: str, fs, city_name: str):
    """
    Robust single-file S3 write:
    local -> put(tmp) -> mv(tmp->final)
    Prevents 0-byte final objects.
    """
    local_tmp = f"/tmp/{city_name}_{uuid.uuid4().hex}.geoparquet"
    out_tmp = out_s3_uri + ".tmp"

    # remove 0-byte leftover at final if any
    try:
        if fs.exists(out_s3_uri) and fs.info(out_s3_uri).get("Size", 0) == 0:
            fs.rm(out_s3_uri)
    except Exception:
        pass

    # write local first
    gdf.to_parquet(local_tmp, engine="pyarrow", index=False)

    # upload then swap into place
    fs.put(local_tmp, out_tmp)
    fs.mv(out_tmp, out_s3_uri)

    try:
        os.remove(local_tmp)
    except Exception:
        pass


def apply_metric_standardization(df):
    """
    Adds m{i}_std from m{i}_raw using the per-metric functions in standardize_metrics.py.
    Expects keys like 'metric_1', 'metric_2', ...
    """
    import re

    for key, std_func in standardization_functions.items():
        m = re.match(r"metric_(\d+)$", str(key))
        if not m:
            continue

        i = int(m.group(1))
        raw_col = f"m{i}_raw"
        std_col = f"m{i}_std"

        if raw_col not in df.columns:
            continue

        df[std_col] = df[raw_col].map_partitions(
            std_func,
            meta=(std_col, "float64")
        )

    return df

def consolidate_city_to_all(city_name: str, YOUR_NAME: str):
    """
    PASS 2:
    Read the per-metric-group outputs, merge safely (k_* prefixed),
    add *_std (per-city), and write ONE consolidated file.
    """
    base = f"{OUTPUT_PATH_RASTER}/{city_name}/{city_name}"

    p_12        = f"{base}_block_metrics_1_2_{YOUR_NAME}"
    p_345101112 = f"{base}_block_metrics_3_4_5_10_11_12_{YOUR_NAME}"
    p_67        = f"{base}_block_metrics_6_7_{YOUR_NAME}"
    p_89        = f"{base}_block_metrics_8_9_{YOUR_NAME}"
    p_k         = f"{base}_block_metrics_k_{YOUR_NAME}.geoparquet"

    out = f"{base}_block_metrics_ALL_{YOUR_NAME}.geoparquet"

    # skip if exists and non-empty
    try:
        if fs.exists(out) and fs.info(out).get("Size", 0) > 0:
            return out
    except Exception:
        pass

    # geometry master
    g_base = dgpd.read_parquet(p_345101112)

    g_12 = dgpd.read_parquet(p_12)
    g_67 = dgpd.read_parquet(p_67)
    g_89 = dgpd.read_parquet(p_89)
    g_k  = dgpd.read_parquet(p_k)

    df = g_base
    df = safe_merge_on_index(df, g_12)
    df = safe_merge_on_index(df, g_67)
    df = safe_merge_on_index(df, g_89)

    # prefix ALL k-block cols to avoid collisions (optimal_point, max_radius, etc.)
    df = safe_merge_on_index(df, g_k, right_prefix="k_")

    # per-city standardization (NOT global)
    df = apply_metric_standardization(df)

    pdf = df.compute()
    gdf = gpd.GeoDataFrame(pdf, geometry="geometry", crs=g_base.crs)

    write_parquet_to_s3_atomic(gdf, out, fs, city_name=city_name)
    return out


for city_name in only_tokens:
    consolidate_city_to_all(city_name, 'sara')